In [ ]:
# ================================================================
# Environment Setup for 01_Build_Dataset.ipynb
# Compatible with Python 3.12 and Google Colab
# ================================================================

!pip install -U --force-reinstall --no-cache-dir \
  numpy==2.0.2 \
  pandas==2.2.2 \
  scipy==1.13.1 \
  pyarrow==18.1.0 \
  duckdb==1.3.2 \
  scikit-learn==1.6.1 \
  statsmodels==0.14.6 \
  lightgbm==4.6.0 \
  optuna==3.6.1 \
  holidays==0.86 \
  fredapi==0.5.2 \
  python-dotenv==1.2.1 \
  matplotlib==3.9.2 \
  seaborn==0.13.2


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

### Dataset Build (End-to-End)

This notebook rebuilds `data/processed/model_ready.parquet` from raw sources:

- Joins COMED hourly load with weather (DuckDB SQL).
- Pulls FRED macro indicators (INDPRO, DCOILWTICO, UNRATE) via `fredapi`.
- Aligns macro to hourly with forward-fill (no look-ahead leakage).
- Engineers features (lags, rolls, calendar, interactions).
- Writes the final artifact for downstream modeling.

If you do not have a FRED API key, skip this notebook and use the cached file shipped in the repo.


In [3]:
# --- Execute end-to-end pipeline with checks & logging ---

import os, sys, time, hashlib, importlib
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

# Repo root (adjust if you moved the repo)
REPO_DIR = Path("/content/drive/")
os.chdir(REPO_DIR)

# Ensure local src package is importable
SRC = REPO_DIR / "src"
if SRC.as_posix() not in sys.path:
    sys.path.append(SRC.as_posix())

# Load env so FRED_API_KEY is available (add .env at repo root)
load_dotenv(dotenv_path=REPO_DIR / ".env")

# Pre-flight: check FRED key early to avoid sys.exit(...) inside run_pipeline
fred_key = os.getenv("FRED_API_KEY", "")
if not fred_key:
    raise RuntimeError(
        "FRED_API_KEY not found. Create a `.env` file at repo root containing:\n"
        'FRED_API_KEY="YOUR_FRED_KEY_HERE"\n'
        "Then re-run this cell."
    )


In [ ]:
RAW = REPO_DIR / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)
expected_files = [
    RAW / "COMED_hourly.csv",
    RAW / "temperature.csv",
    RAW / "humidity.csv",
    RAW / "pressure.csv",
    RAW / "wind_speed.csv",
    RAW / "wind_direction.csv",
    RAW / "weather_description.csv",
]
for f in expected_files:
    print(f"{f.name:<24} {'✅' if f.exists() else '❌'}")


In [ ]:
# Import pipeline modules AFTER sys.path and env are set
import src.sql_build as sql_build
import src.make_dataset as make_dataset

# Hot-reload during iterative dev
importlib.reload(sql_build)
importlib.reload(make_dataset)

# Run pipeline
t0 = time.perf_counter()
final_path = make_dataset.run_pipeline(preview_rows=10)
elapsed = time.perf_counter() - t0

print(f"\n⏱️ Pipeline runtime: {elapsed:.2f} s")
print(f"📦 Final artifact:  {final_path}")

# Validate artifact (read via pyarrow; fallback to fastparquet depending on preferance)
final_path = Path(final_path)
assert final_path.exists(), f"Expected artifact missing: {final_path}"

try:
    df = pd.read_parquet(final_path)  # default engine (pyarrow)
except Exception as e:
    print("⚠️ pyarrow read failed, trying fastparquet. Error was:\n", e)
    df = pd.read_parquet(final_path, engine="fastparquet")

# Quick summary
df["ts"] = pd.to_datetime(df["ts"], utc=True)
df = df.sort_values("ts").drop_duplicates(subset="ts")

print(f"\n✅ Loaded processed dataset: {len(df):,} rows × {df.shape[1]} columns")
print(f"   Time range: {df['ts'].min()} → {df['ts'].max()}")
print("   Sample columns:", list(df.columns[:8]), "...")

# Missingness snapshot for top engineered features
miss = df.isna().mean().sort_values(ascending=False).head(10)
print("\n🔎 Missingness (top 10):\n", miss)

# Deterministic file hash (useful for caching / reproducibility notes)
md5 = hashlib.md5(final_path.read_bytes()).hexdigest()
print(f"\n🔐 Artifact MD5: {md5}")